In [28]:
import pandas as pd
import statsmodels.api as sm


# ============================================================
# 0. Cargar base
# ============================================================

df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/base_distrital_2018_2022_con_candidatos.xlsx")


# ============================================================
# 1. GANADORES 2018 Y 2022 (por DNI y por PARTIDO)
# ============================================================

# ------- Ganadores 2018 -------
g18 = df[df["año"] == 2018].sort_values(
    ["ubigeo", "total_votos"], ascending=[True, False]
).groupby("ubigeo").nth(0).reset_index()

g18_dni = g18[["ubigeo", "dni"]].rename(columns={"dni": "dni_ganador_2018"})
g18_partido = g18[["ubigeo", "organizacion_politica"]].rename(
    columns={"organizacion_politica": "partido_ganador_2018"}
)

# ------- Ganadores 2022 -------
g22 = df[df["año"] == 2022].sort_values(
    ["ubigeo", "total_votos"], ascending=[True, False]
).groupby("ubigeo").nth(0).reset_index()

g22_dni = g22[["ubigeo", "dni"]].rename(columns={"dni": "dni_ganador_2022"})
g22_partido = g22[["ubigeo", "organizacion_politica"]].rename(
    columns={"organizacion_politica": "partido_ganador_2022"}
)


In [29]:
# ============================================================
# 2. TURNOVER por CANDIDATO y por PARTIDO
# ============================================================

# --- Turnover CANDIDATO ---
tm_candidato = g18_dni.merge(g22_dni, on="ubigeo", how="left")
tm_candidato["turnover_candidato"] = (
    tm_candidato["dni_ganador_2018"] == tm_candidato["dni_ganador_2022"]
).astype(int)

# --- Turnover PARTIDO ---
tm_partido = g18_partido.merge(g22_partido, on="ubigeo", how="left")
tm_partido["turnover_partido"] = (
    tm_partido["partido_ganador_2018"] == tm_partido["partido_ganador_2022"]
).astype(int)


In [30]:
# ============================================================
# 3. POSICIÓN DEL INCUMBENT (posición del ganador 2018)
# ============================================================

incumbent_pos = g18[["ubigeo", "orden_aparicion"]].rename(
    columns={"orden_aparicion": "pos_incumbent"}
)

In [31]:
# ============================================================
# 4. DATA FINAL PARA REGRESIONES
# ============================================================

final = (
    tm_candidato
    .merge(tm_partido, on="ubigeo")
    .merge(incumbent_pos, on="ubigeo")
)

# Quitamos missing
final = final.dropna(subset=["pos_incumbent"])


In [32]:
# ============================================================
# 5. REGRESIÓN LPM: turnover por PARTIDO
# ============================================================

X1 = sm.add_constant(final["pos_incumbent"])
y1 = final["turnover_partido"]

lpm_partido = sm.OLS(y1, X1).fit()

print("\n================ LPM: TURNOVER POR PARTIDO ================\n")
print(lpm_partido.summary())


================ LPM: TURNOVER POR PARTIDO ================

                            OLS Regression Results                            
Dep. Variable:       turnover_partido   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     13.53
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           0.000242
Time:                        15:33:42   Log-Likelihood:                 5.1835
No. Observations:                1678   AIC:                            -6.367
Df Residuals:                    1676   BIC:                             4.484
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------

In [33]:
# ============================================================
# 6. REGRESIÓN LPM: turnover por CANDIDATO (DNI)
# ============================================================

X2 = sm.add_constant(final["pos_incumbent"])
y2 = final["turnover_candidato"]

lpm_candidato = sm.OLS(y2, X2).fit()

print("\n================ LPM: TURNOVER POR CANDIDATO ================\n")
print(lpm_candidato.summary())


================ LPM: TURNOVER POR CANDIDATO ================

                            OLS Regression Results                            
Dep. Variable:     turnover_candidato   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.7070
Date:                Mon, 24 Nov 2025   Prob (F-statistic):              0.401
Time:                        15:33:45   Log-Likelihood:                 3268.7
No. Observations:                1678   AIC:                            -6533.
Df Residuals:                    1676   BIC:                            -6523.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------

In [37]:

# ============================================================
# 0. Cargar base
# ============================================================

df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/base_distrital_2018_2022_con_candidatos.xlsx")

# ============================================================
# 1. GANADORES 2018 Y 2022 (por DNI y por PARTIDO)
# ============================================================

# ------- Ganadores 2018 -------
g18 = df[df["año"] == 2018].sort_values(
    ["ubigeo", "total_votos"], ascending=[True, False]
).groupby("ubigeo").nth(0).reset_index()

g18_dni = g18[["ubigeo", "dni"]].rename(columns={"dni": "dni_ganador_2018"})
g18_partido = g18[["ubigeo", "organizacion_politica"]].rename(
    columns={"organizacion_politica": "partido_ganador_2018"}
)

# ------- Ganadores 2022 -------
g22 = df[df["año"] == 2022].sort_values(
    ["ubigeo", "total_votos"], ascending=[True, False]
).groupby("ubigeo").nth(0).reset_index()

g22_dni = g22[["ubigeo", "dni"]].rename(columns={"dni": "dni_ganador_2022"})
g22_partido_winner = g22[["ubigeo", "organizacion_politica"]].rename(
    columns={"organizacion_politica": "partido_ganador_2022"}
)

# ============================================================
# 2. TURNOVER por CANDIDATO y por PARTIDO
# ============================================================

# --- Turnover CANDIDATO ---
tm_candidato = g18_dni.merge(g22_dni, on="ubigeo", how="left")
tm_candidato["turnover_candidato"] = (
    tm_candidato["dni_ganador_2018"] == tm_candidato["dni_ganador_2022"]
).astype(int)

# --- Turnover PARTIDO (según ganador) ---
tm_partido = g18_partido.merge(g22_partido_winner, on="ubigeo", how="left")
tm_partido["turnover_partido"] = (
    tm_partido["partido_ganador_2018"] == tm_partido["partido_ganador_2022"]
).astype(int)

# ============================================================
# 3. POSICIÓN DEL INCUMBENT (ganador 2018)
# ============================================================

incumbent_pos = g18[["ubigeo", "orden_aparicion"]].rename(
    columns={"orden_aparicion": "pos_incumbent"}
)

# ============================================================
# 4. DATA BASE PARA REGRESIONES (antes del filtro)
# ============================================================

final = (
    tm_candidato
    .merge(tm_partido, on="ubigeo")
    .merge(incumbent_pos, on="ubigeo")
)

# ============================================================
# 5. IDENTIFICAR SI EL PARTIDO GANADOR 2018 VUELVE A PARTICIPAR EN 2022
# ============================================================

# Todos los partidos que compiten en 2022 por distrito
parties_2022 = df[df["año"] == 2022][["ubigeo", "organizacion_politica"]].drop_duplicates()
parties_2022["participa_2022"] = 1

# Ver si el partido_ganador_2018 aparece entre los que compiten en 2022
final = final.merge(
    parties_2022,
    left_on=["ubigeo", "partido_ganador_2018"],
    right_on=["ubigeo", "organizacion_politica"],
    how="left"
).drop(columns=["organizacion_politica"])

final["participa_2022"] = final["participa_2022"].fillna(0)

# ============================================================
# 6. FILTRAR: solo casos donde el partido ganador 2018 participa de nuevo
# ============================================================

final_restricted = final[final["participa_2022"] == 1].copy()

# (Opcional) ver cuántas observaciones quedan
print("Observaciones totales:", len(final))
print("Observaciones donde el partido 2018 participa en 2022:", len(final_restricted))



Observaciones totales: 1678
Observaciones donde el partido 2018 participa en 2022: 579


In [39]:
# ============================================================
# 7. REGRESIÓN LPM: turnover por PARTIDO (muestra restringida)
# ============================================================

final_restricted = final_restricted.dropna(subset=["pos_incumbent"])

X1 = sm.add_constant(final_restricted["pos_incumbent"])
y1 = final_restricted["turnover_partido"]

lpm_partido = sm.OLS(y1, X1).fit()

print("\n================ LPM (Muestra restringida): TURNOVER POR PARTIDO ================\n")
print(lpm_partido.summary())





================ LPM (Muestra restringida): TURNOVER POR PARTIDO ================

                            OLS Regression Results                            
Dep. Variable:       turnover_partido   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     11.01
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           0.000962
Time:                        15:34:47   Log-Likelihood:                -263.89
No. Observations:                 579   AIC:                             531.8
Df Residuals:                     577   BIC:                             540.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------

In [40]:
# ============================================================
# 8. REGRESIÓN LPM: turnover por CANDIDATO (DNI) (muestra restringida)
# ============================================================

X2 = sm.add_constant(final_restricted["pos_incumbent"])
y2 = final_restricted["turnover_candidato"]

lpm_candidato = sm.OLS(y2, X2).fit()

print("\n================ LPM (Muestra restringida): TURNOVER POR CANDIDATO ================\n")
print(lpm_candidato.summary())


================ LPM (Muestra restringida): TURNOVER POR CANDIDATO ================

                            OLS Regression Results                            
Dep. Variable:     turnover_candidato   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.3032
Date:                Mon, 24 Nov 2025   Prob (F-statistic):              0.582
Time:                        15:34:49   Log-Likelihood:                 1020.7
No. Observations:                 579   AIC:                            -2037.
Df Residuals:                     577   BIC:                            -2029.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------